In [1]:
!nvidia-smi

Fri May  1 20:42:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import os
PROJECT = '/content/drive/MyDrive/llm-detection'
os.makedirs(f'{PROJECT}/models/seam_detector_v3', exist_ok=True)
print('PROJECT:', PROJECT)
print('Zip present?', os.path.exists(f'{PROJECT}/llm-detection-colab.zip'))
print('Contents:', sorted(os.listdir(PROJECT))[:20])

os.chdir(PROJECT)
# !unzip -o llm-detection-colab.zip
!ls -la

!pip install -q --upgrade pip
!pip install -q "transformers==4.49.0" "tokenizers==0.21.4" "safetensors==0.4.5" "huggingface_hub==0.26.5"
!pip install -q peft "pytorch-crf==0.7.2"
!pip install -q bitsandbytes
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0))
print('bf16 supported:', torch.cuda.is_bf16_supported())

!python -c "import sys; sys.path.insert(0, '.'); from train_seam_detector import SeamDetector, FeatureFGM, CrossLayerAttentionFusionV2; print('imports OK')"
!pip install -q --upgrade "torchao>=0.16.0"

import torchao
print('torchao version:', torchao.__version__)

PROJECT: /content/drive/MyDrive/llm-detection
Zip present? True
Contents: ['__pycache__', 'evaluate_seam_detector_v3.py', 'llm-detection-colab.zip', 'models', 'predict_document.py', 'test.csv', 'train.csv', 'train_seam_detector.py', 'val.csv']
total 367383
-rw------- 1 root root     19581 May  1 16:35 evaluate_seam_detector_v3.py
-rw------- 1 root root  84482011 May  1 09:27 llm-detection-colab.zip
drwx------ 3 root root      4096 May  1 09:31 models
-rw------- 1 root root     20612 May  1 16:38 predict_document.py
drwx------ 2 root root      4096 May  1 09:32 __pycache__
-rw------- 1 root root  29136396 May  1 11:20 test.csv
-rw------- 1 root root 233209482 May  1 11:20 train.csv
-rw------- 1 root root     97696 May  1 17:16 train_seam_detector.py
-rw------- 1 root root  29224349 May  1 11:20 val.csv
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 1.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This

torchao version: 0.17.0


In [3]:
# Pre-download DeBERTa-v3-Large to Drive (one-time, ~2-3 minutes)
import os
from huggingface_hub import snapshot_download

LOCAL_MODEL_DIR = f'{PROJECT}/models/deberta-v3-large'

if not os.path.exists(f'{LOCAL_MODEL_DIR}/model.safetensors'):
    print(f"Downloading microsoft/deberta-v3-large to {LOCAL_MODEL_DIR}...")
    snapshot_download(
        repo_id='microsoft/deberta-v3-large',
        local_dir=LOCAL_MODEL_DIR,
        local_dir_use_symlinks=False,   # actual files, not symlinks (Drive-friendly)
        max_workers=4,
    )
    print('Download complete.')
else:
    print(f'Already downloaded at {LOCAL_MODEL_DIR}')

print('Files:')
!ls -la {LOCAL_MODEL_DIR}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:834: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Download complete.
Files:
total 3109410
drwx------ 3 root root       4096 May  1 09:43 .cache
-rw------- 1 root root        580 May  1 09:43 config.json
-rw------- 1 root root        560 May  1 09:43 generator_config.json
-rw------- 1 root root       1175 May  1 09:43 .gitattributes
-rw------- 1 root root  873673253 May  1 09:43 pytorch_model.bin
-rw------- 1 root root  571293153 May  1 09:43 pytorch_model.generator.bin
-rw------- 1 root root       3266 May  1 09:43 README.md
-rw------- 1 root root    2464616 May  1 09:43 spm.model
-rw------- 1 root root 1736592160 May  1 09:43 tf_model.h5
-rw------- 1 root root         52 May  1 09:43 tokenizer_config.json


In [4]:
# 1. Verify the step-500 save is on disk
!ls models/seam_detector_v3/step_000500/
!cat models/seam_detector_v3/step_000500/training_state.json


lora_adapter   rng_state.pth	   training_state.json
optimizer.pth  training_args.json
{
  "epoch_completed": 0,
  "step_within_epoch": 500,
  "global_update": 500,
  "best_seam_offset": 36.66331125827814,
  "best_f1_at_5": 0.25438072481083235,
  "epochs_since_improvement": 0,
  "saved_at": "after_mid_epoch_validation"
}

In [5]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.chdir(PROJECT)

LOCAL_MODEL = '/content/drive/MyDrive/llm-detection/models/deberta-v3-large'

!PYTORCH_ALLOC_CONF=expandable_segments:True python train_seam_detector.py \
    --resume \
    --train-csv train.csv --val-csv val.csv --test-csv test.csv \
    --output-dir models/seam_detector_v3 \
    --model-name {LOCAL_MODEL} \
    --num-epochs 6 \
    --batch-size 32 --gradient-accumulation-steps 1 \
    --no-gradient-checkpointing \
    --torch-compile-mode default \
    --patience 3 \
    --validate-every-steps 1000 \
    --save-every-steps 500 \
    --log-every 10 \
    --gate-monitor --dataloader-workers 6 \
    --head-lr 5e-4 \
    --fgm-epsilon 0.2 \
    --max-grad-norm 0.5 \
    --min-p-1to0 0.15 \
    --loss-spike-factor 5.0 \
    --lambda-focal 0.3 \
    --focal-gamma 2.0 \
    --focal-seam-alpha 0.75


2026-05-01 20:44:35.728436: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-01 20:44:35.799005: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Device: cuda  |  bf16=True  |  LoRA=True  |  checkpointing=False  |  FGM=True
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of 

In [6]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.chdir(PROJECT)

LOCAL_MODEL = '/content/drive/MyDrive/llm-detection/models/deberta-v3-large'

!PYTORCH_ALLOC_CONF=expandable_segments:True python train_seam_detector.py \
    --resume \
    --train-csv train.csv --val-csv val.csv --test-csv test.csv \
    --output-dir models/seam_detector_v3 \
    --model-name {LOCAL_MODEL} \
    --num-epochs 6 \
    --batch-size 16 --gradient-accumulation-steps 2 \
    --no-gradient-checkpointing \
    --torch-compile-mode default \
    --patience 3 \
    --validate-every-steps 1000 \
    --save-every-steps 500 \
    --log-every 10 \
    --gate-monitor --dataloader-workers 6 \
    --head-lr 5e-4 \
    --fgm-epsilon 0.2 \
    --max-grad-norm 0.5 \
    --min-p-1to0 0.15 \
    --loss-spike-factor 5.0 \
    --lambda-focal 0.3 \
    --focal-gamma 2.0 \
    --focal-seam-alpha 0.75


2026-05-01 21:23:07.678881: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-01 21:23:07.749698: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Device: cuda  |  bf16=True  |  LoRA=True  |  checkpointing=False  |  FGM=True
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of 